# Phase 4 Agent Demo

This is the clean demo notebook for the LangGraph agent.

It shows four paths:

- clear positive image -> cautious report
- normal image -> no disease-specific draft
- Other/low-confidence image -> no disease-specific draft
- bad path -> input refusal

It also includes one critic stress test where the draft intentionally claims something unsupported.

## Setup

In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.agent.run import run_agent
from src.agent.nodes import critic_node

print(PROJECT_ROOT)

E:\Project\RadScribe


## Run The Four Main Paths

These are fixed examples, not random samples.

In [2]:
cases = [
    {
        "case": "clear_positive",
        "image_path": PROJECT_ROOT / "data" / "processed" / "images_224" / "797_IM-2332-1001.dcm.png",
        "expected": "cautious disease-specific report",
    },
    {
        "case": "normal",
        "image_path": PROJECT_ROOT / "data" / "processed" / "images_224" / "3528_IM-1725-2002.dcm.png",
        "expected": "no disease-specific draft",
    },
    {
        "case": "other_low_confidence",
        "image_path": PROJECT_ROOT / "data" / "processed" / "images_224" / "1131_IM-0088-0001-0002.dcm.png",
        "expected": "no disease-specific draft",
    },
    {
        "case": "bad_path_guardrail",
        "image_path": PROJECT_ROOT / "data" / "processed" / "images_224" / "not_real.png",
        "expected": "input refused before vision",
    },
]

rows = []
for item in cases:
    result = run_agent(item["image_path"])
    rows.append(
        {
            "case": item["case"],
            "expected": item["expected"],
            "guardrail_passed": result.get("guardrail_passed"),
            "vision_status": result.get("vision_status"),
            "main_findings": result.get("main_findings"),
            "borderline_findings": result.get("borderline_findings"),
            "can_draft": result.get("can_draft"),
            "best_retrieval_score": round(float(result.get("best_retrieval_score", 0.0)), 3),
            "trace_path": result.get("trace_path"),
            "final_report": result.get("final_report"),
        }
    )

runs_df = pd.DataFrame(rows)
runs_df[[
    "case",
    "expected",
    "guardrail_passed",
    "vision_status",
    "main_findings",
    "borderline_findings",
    "can_draft",
    "best_retrieval_score",
    "trace_path",
]]

,case,expected,guardrail_passed,vision_status,main_findings,borderline_findings,can_draft,best_retrieval_score,trace_path
0,clear_positive,cautious disease-specific report,True,high_confidence,"[Cardiomegaly, Edema]",[Consolidation / Pneumonia],True,0.765,E:\Project\RadScribe\outputs\agent\traces\2026...
1,normal,no disease-specific draft,True,no_finding_above_threshold,[],[],None,0.000,E:\Project\RadScribe\outputs\agent\traces\2026...
2,other_low_confidence,no disease-specific draft,True,no_finding_above_threshold,[],[],None,0.000,E:\Project\RadScribe\outputs\agent\traces\2026...
3,bad_path_guardrail,input refused before vision,False,input_refused,[],[],False,0.000,E:\Project\RadScribe\outputs\agent\traces\2026...


## Read The Final Reports

In [3]:
for row in rows:
    print("=" * 80)
    print(row["case"])
    print(row["final_report"])
    print("trace:", row["trace_path"])


clear_positive
**Findings:**  
Model suggests cardiomegaly, indicated by an enlarged cardiac silhouette, which may reflect underlying heart strain or dysfunction. Model also suggests edema, potentially causing respiratory symptoms such as shortness of breath and cough. Borderline findings may include possible consolidation or pneumonia.

**Impression:**  
- Cardiomegaly and edema are highly probable findings.  
- Possible consolidation or pneumonia may warrant further evaluation.

**Evidence:**  
(cardiomegaly_001, cardiomegaly_002, edema_001)

Educational prototype. Not a medical device, not a diagnosis. For research use only; consult a qualified radiologist.
trace: E:\Project\RadScribe\outputs\agent\traces\20260822T050941Z_797_IM-2332-1001.dcm.json
normal
No finding above the model confidence threshold. No disease-specific draft was generated.

Educational prototype. Not a medical device, not a diagnosis. For research use only; consult a qualified radiologist.
trace: E:\Project\RadSc

## Check Saved Trace Contains Disclaimer

This checks the report path and the no-draft/refusal paths.

In [4]:
DISCLAIMER = "Educational prototype. Not a medical device, not a diagnosis. For research use only; consult a qualified radiologist."

trace_checks = []
for row in rows:
    trace = json.loads(Path(row["trace_path"]).read_text(encoding="utf-8"))
    trace_checks.append(
        {
            "case": row["case"],
            "has_disclaimer": trace["final_report"].endswith(DISCLAIMER),
            "trace_path": row["trace_path"],
        }
    )

pd.DataFrame(trace_checks)

,case,has_disclaimer,trace_path
0,clear_positive,True,E:\Project\RadScribe\outputs\agent\traces\2026...
1,normal,True,E:\Project\RadScribe\outputs\agent\traces\2026...
2,other_low_confidence,True,E:\Project\RadScribe\outputs\agent\traces\2026...
3,bad_path_guardrail,True,E:\Project\RadScribe\outputs\agent\traces\2026...


## Critic Stress Test

This is not a natural agent run. I intentionally give the critic a bad draft that claims a large pneumothorax while the evidence comes from the clear-positive trace. The critic should mark it unsupported.

In [5]:
clear_positive_trace = json.loads(Path(runs_df.loc[runs_df["case"] == "clear_positive", "trace_path"].iloc[0]).read_text(encoding="utf-8"))

bad_draft_state = {
    "draft_report": "Findings: Model suggests possible cardiomegaly. There is also a large pneumothorax.\n\nImpression: Possible cardiomegaly and large pneumothorax.",
    "evidence": clear_positive_trace["evidence"],
}

critic_test = critic_node(bad_draft_state)
critic_test["critic_result"]

{'supported': False,
 'missing_evidence': ['large pneumothorax'],
 'safety_note': 'The draft mentions a large pneumothorax, which is not supported by the evidence.'}

## Short Takeaway

Fill this in after running:

- Clear positive produced a cautious report and saved a trace.
- Normal and Other/low-confidence produced no disease-specific draft.
- Bad path was refused before vision.
- Every final report had the disclaimer.
- The critic stress test returned `supported = ___` and flagged `___`.